# Line balancing of a packaging line — discrete-event simulation (SimPy)

**Author:** Esteban López · Industrial Engineer  
**Tools:** SimPy (discrete-event simulation), queueing theory, Lean line balancing.

## Project goal
Most shop-floor improvement work has to happen on site. This project shows the **method** I would apply, on a
digital twin: model a 6-station packaging/conditioning line with **SimPy**, find the **bottleneck**, compare line
throughput against **takt time**, and quantify the gain from **balancing the line** — in throughput, lead time and WIP.

> Self-contained and **reproducible** (`pip install simpy`). The data are the line parameters, defined explicitly below — this is a *simulation*, so the model is the experiment.

In [ ]:
import simpy, random
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})
NAVY, CYAN, GOLD, SOFT = '#0b1f3a', '#16b3c7', '#f2c14e', '#9fb3c8'

DEMAND = 80.0                  # units required per hour
TAKT   = 3600 / DEMAND         # seconds per unit the line must hit (45 s)
print(f'Takt time = {TAKT:.0f} s/unit  (demand {DEMAND:.0f} u/h)')

## 1. DEFINE / MEASURE — The line
Six stations in series. Mean cycle times (seconds). Station 3 is heavy — the suspected bottleneck.

In [ ]:
base_times = [30, 35, 55, 33, 40, 28]      # s/unit per station (S1..S6)
stations   = [f'S{i+1}' for i in range(len(base_times))]
print('Total work content:', sum(base_times), 's  |  theoretical min stations at takt:',
      int(np.ceil(sum(base_times)/TAKT)))
print('Bottleneck:', stations[int(np.argmax(base_times))], '=', max(base_times), 's  > takt', int(TAKT), 's')

## 2. The SimPy model
A serial line: each station is a single-capacity resource; parts arrive at takt and flow through every station.
Service times are exponential around each mean (process variability).

In [ ]:
def run_line(times, sim_hours=40, seed=1):
    random.seed(seed)
    env = simpy.Environment()
    res = [simpy.Resource(env, capacity=1) for _ in times]
    leads, done, in_system = [], [0], [0]

    def part(pid, t_in):
        in_system[0] += 1
        for i, r in enumerate(times):
            with res[i].request() as req:
                yield req
                yield env.timeout(random.expovariate(1.0 / times[i]))
        leads.append(env.now - t_in); done[0] += 1; in_system[0] -= 1

    def source():
        pid = 0
        while True:
            yield env.timeout(random.expovariate(1.0 / TAKT))   # arrivals at takt
            env.process(part(pid, env.now)); pid += 1

    # sample WIP periodically
    wip = []
    def monitor():
        while True:
            wip.append(in_system[0]); yield env.timeout(60)
    env.process(source()); env.process(monitor())
    env.run(until=sim_hours * 3600)
    thr = done[0] / sim_hours
    return dict(throughput=thr, lead_s=np.mean(leads) if leads else np.nan,
                wip=np.mean(wip[5:]), bottleneck_cap=3600 / max(times))

base = run_line(base_times)
print(f"BEFORE  throughput={base['throughput']:.1f} u/h (cap {base['bottleneck_cap']:.0f}) "
      f"| lead={base['lead_s']/60:.1f} min | WIP={base['wip']:.1f}")
print('Demand is', DEMAND, 'u/h -> the line CANNOT keep up: the bottleneck caps it.')

## 3. ANALYZE — Bottleneck vs takt

In [ ]:
util = [t / TAKT for t in base_times]   # utilisation at demand = cycle time / takt
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(stations, base_times, color=[GOLD if t > TAKT else CYAN for t in base_times], edgecolor='white')
for i, t in enumerate(base_times):
    ax.text(i, t + 0.8, f'{t}s', ha='center', fontweight='bold', color=NAVY)
ax.axhline(TAKT, color='grey', ls='--'); ax.text(5.1, TAKT + 0.6, f'Takt {TAKT:.0f}s', ha='right', color='grey')
ax.set_ylabel('Cycle time (s)'); ax.set_title('Station cycle times vs takt — bottleneck at S3', fontweight='bold', color=NAVY)
plt.tight_layout(); plt.show()
for s, u in zip(stations, util):
    flag = '  <-- OVERLOADED' if u > 1 else ''
    print(f'{s}: utilisation {u:.2f}{flag}')

**Finding:** S3 needs 55 s but takt is 45 s → utilisation 1.22 (> 1). One overloaded station throttles the whole line; throughput is capped well below demand. The fix is to **rebalance work** so no station exceeds takt.

## 4. IMPROVE — Rebalance the line
Redistribute ~13 s of S3's work to lighter stations (same total work content) so every station sits below takt.

In [ ]:
bal_times = [38, 38, 42, 38, 37, 28]    # same total work (221 s), no station over takt
assert sum(bal_times) == sum(base_times)
bal = run_line(bal_times)
print(f"AFTER   throughput={bal['throughput']:.1f} u/h (cap {bal['bottleneck_cap']:.0f}) "
      f"| lead={bal['lead_s']/60:.1f} min | WIP={bal['wip']:.1f}")

fig, ax = plt.subplots(figsize=(9, 5)); x = np.arange(6); w = 0.38
ax.bar(x - w/2, base_times, w, label='Before', color=SOFT, edgecolor='white')
ax.bar(x + w/2, bal_times, w, label='After', color=CYAN, edgecolor='white')
ax.axhline(TAKT, color=GOLD, ls='--'); ax.text(5.3, TAKT + 0.6, f'Takt {TAKT:.0f}s', ha='right', color='#b8902a')
ax.set_xticks(x); ax.set_xticklabels(stations); ax.set_ylabel('Cycle time (s)')
ax.set_title('Line balancing — every station now under takt', fontweight='bold', color=NAVY); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
gain = (bal['throughput'] - base['throughput']) / base['throughput'] * 100
print(f"Throughput: {base['throughput']:.0f} -> {bal['throughput']:.0f} u/h  (+{gain:.0f}%)")
print(f"Lead time : {base['lead_s']/60:.1f} -> {bal['lead_s']/60:.1f} min")
print(f"WIP       : {base['wip']:.0f} -> {bal['wip']:.0f} units")
print(f"Demand met: {'NO' if base['throughput'] < DEMAND else 'yes'} -> {'YES' if bal['throughput'] >= DEMAND else 'no'}")

## 5. CONTROL — Sustain the balance

1. **Standardised work** at each station so cycle times stay near the balanced target.
2. **Visual takt** on the line; react when a station drifts above it (the new bottleneck).
3. **Re-balance when demand changes** — takt is set by demand, so recompute when the order rate moves.
4. **Attack variability**, not just the mean: SimPy shows that high variability inflates WIP and lead time even when the means fit under takt.
5. **PDCA**: re-simulate proposed changes before touching the physical line.

*Why simulate first:* the model let me test the rebalancing and quantify throughput, lead time and WIP **before** any change on the floor — cheap, fast and risk-free.